# HydrAI — Symbolic Regression Distillation (any NN teacher)

Distil a trained HydrAI surrogate into closed-form symbolic expressions using
[PySR](https://github.com/MilesCranmer/PySR). Supported teachers:

| `TEACHER_STEM` | Source | Model class | Mode |
|---|---|---|---|
| `simple_nn_full_profile` | Main_6 | `SimpleNN` | full profile (6 + z/L inputs) |
| `pinn_pfr` | Main_7 | `PINNPFR` | full profile (6 + z/L inputs) |

**Pipeline:** teacher NN trained → **Main_8 (this notebook)** distils it → equations
used in Main_10 GP-BO optimisation and/or CFD embedding.

Set `TEACHER_STEM` in §2 and run all cells. Prerequisites: the relevant source notebook
must have been run with `IF_MODEL_EXPORT = True`.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ════════════════════════════════════════════════════════════════════════════
import os
import sys
import json
import joblib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import r2_score, mean_absolute_error

try:
    from pysr import PySRRegressor
    PYSR_AVAILABLE = True
except ImportError:
    PYSR_AVAILABLE = False
    print('[warn] PySR not installed. Run: pip install pysr')

PROJECT_ROOT = Path(os.getcwd())
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models import SimpleNN, PINNPFR
from src.utils.plot_style import setup_matplotlib
setup_matplotlib()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2. PATHS & FLAGS
# ════════════════════════════════════════════════════════════════════════════
# Edit configs/ml/main8_symbolic_regression_config.json to change these
# (edit and re-run this cell; no kernel restart needed).
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'ml' / 'main8_symbolic_regression_config.json'
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        _cfg = json.load(f)
else:
    _cfg = {}
    print(f'[WARN] Config not found: {CONFIG_PATH}. Using defaults.')

IF_PLOT_SHOWN  = _cfg.get('if_plot_shown', True)
IF_PLOT_EXPORT = _cfg.get('if_plot_export', True)
IF_SR_EXPORT   = _cfg.get('if_sr_export', True)

# PySR budget — increase for production runs
SR_N_ITERATIONS  = _cfg.get('sr_n_iterations', 100)
SR_POPULATIONS   = _cfg.get('sr_populations', 30)
SR_POPULATION_SIZE = _cfg.get('sr_population_size', 33)
SR_MAXSIZE       = _cfg.get('sr_maxsize', 25)
SR_PROCS         = _cfg.get('sr_procs', 0)

N_DISTILL_SAMPLES = _cfg.get('n_distill_samples', 5_000)

RANDOM_STATE = _cfg.get('random_state', 42)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ── Teacher selection ──────────────────────────────────────────────────────
# Options:
#   'simple_nn_full_profile'  → Main_6 SimpleNN, full axial profile (6+z inputs)
#   'pinn_pfr'                → Main_7 PINNPFR,  full axial profile (6+z inputs)
TEACHER_STEM = _cfg.get('teacher_stem', 'simple_nn_full_profile')

_stem_to_export = {
    'simple_nn_full_profile': ('sr_full_profile', 'sr_full_profile'),
    'pinn_pfr':               ('sr_pinn',         'sr_pinn'),
}
_export_subdir, EXPORT_STEM = _stem_to_export.get(TEACHER_STEM, ('sr_custom', 'sr_custom'))
IS_PROFILE_MODEL = TEACHER_STEM in ('simple_nn_full_profile', 'pinn_pfr')

MODELS_DIR = PROJECT_ROOT / 'models'
FIGS_DIR   = PROJECT_ROOT / 'outputs' / 'figures' / 'Main_8_symbolic_regression'
EXPORT_DIR = PROJECT_ROOT / 'models' / _export_subdir
FIGS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Teacher stem:    {TEACHER_STEM}')
print(f'Profile model:   {IS_PROFILE_MODEL}')
print(f'Export dir:      {EXPORT_DIR}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD TEACHER MODEL
# ════════════════════════════════════════════════════════════════════════════
manifest_path = MODELS_DIR / TEACHER_STEM / f'{TEACHER_STEM}_manifest.json'
_source_nb = {'simple_nn_full_profile': 'Main_6', 'pinn_pfr': 'Main_7'}
if not manifest_path.exists():
    raise FileNotFoundError(
        f'{manifest_path} not found — run {_source_nb.get(TEACHER_STEM, "source notebook")} '
        f'with IF_MODEL_EXPORT=True first.'
    )

with open(manifest_path) as f:
    manifest = json.load(f)

arch        = manifest['architecture']
inlet_cols  = manifest.get('inlet_cols') or manifest.get('feature_cols', [])
target_cols = manifest['target_cols']
print(f'Teacher stem:   {TEACHER_STEM}')
print(f'Trained at:     {manifest.get("run_at", "unknown")}')
print(f'Inputs  ({len(inlet_cols)}): {inlet_cols}')
print(f'Outputs ({len(target_cols)}): {target_cols}')

scalers  = joblib.load(MODELS_DIR / TEACHER_STEM / f'{TEACHER_STEM}_scalers.joblib')
scaler_X = scalers['scaler_X']
scaler_y = scalers.get('scaler_y')

# Instantiate the correct model class
if TEACHER_STEM == 'pinn_pfr':
    teacher = PINNPFR(
        arch['in_features'], arch['h1'], arch['h2'], arch['h3'],
        arch['out_features'], dropout=arch['dropout']
    )
else:
    teacher = SimpleNN(
        arch['in_features'], arch['h1'], arch['h2'], arch['h3'],
        arch['out_features'], dropout=arch['dropout']
    )

teacher.load_state_dict(torch.load(
    MODELS_DIR / TEACHER_STEM / f'{TEACHER_STEM}_state_dict.pt',
    map_location='cpu', weights_only=True,
))
teacher.eval()
print(f'Loaded {teacher.__class__.__name__}: {sum(p.numel() for p in teacher.parameters()):,} params')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 4. GENERATE DISTILLATION DATASET
# ════════════════════════════════════════════════════════════════════════════
data_files = sorted((PROJECT_ROOT / 'data' / 'processed').glob('features_targets_*.pkl'))
if not data_files:
    raise FileNotFoundError('No features_targets_*.pkl in data/processed. Run Main_3.')
df_all = pd.read_pickle(data_files[-1])
print(f'Loaded {len(df_all):,} rows from {data_files[-1].name}')

avail_inlet = [c for c in inlet_cols if c in df_all.columns]
if len(avail_inlet) < len(inlet_cols):
    missing = set(inlet_cols) - set(avail_inlet)
    raise KeyError(f'Feature columns missing from processed data: {missing}')

# Profile models include z_position_m / relative_position as inputs.
# Sample from full dataset rows (each row is a unique (run, z) pair).
X_pool = df_all[inlet_cols].dropna().values
print(f'Profile mode: sampling from {len(X_pool):,} (run, z) rows')

replace = len(X_pool) < N_DISTILL_SAMPLES
idx = np.random.choice(len(X_pool), size=N_DISTILL_SAMPLES, replace=replace)
X_distill = X_pool[idx]

X_s = scaler_X.transform(X_distill)
with torch.no_grad():
    y_s = teacher(torch.tensor(X_s, dtype=torch.float32)).numpy()
y_distill = scaler_y.inverse_transform(y_s) if scaler_y is not None else y_s

df_distill = pd.DataFrame(X_distill, columns=inlet_cols)
df_distill[target_cols] = y_distill
print(f'Distillation dataset: {df_distill.shape}')
df_distill.describe().round(3)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5. SYMBOLIC REGRESSION WITH PYSR
# ════════════════════════════════════════════════════════════════════════════
# We run one multi-output PySR fit for all targets simultaneously.
# PySR finds one equation per output independently.

if not PYSR_AVAILABLE:
    raise RuntimeError('Install PySR: pip install pysr')

X_sr = df_distill[inlet_cols].values
y_sr = df_distill[target_cols].values

sr_model = PySRRegressor(
    niterations     = SR_N_ITERATIONS,
    populations     = SR_POPULATIONS,
    population_size = SR_POPULATION_SIZE,
    maxsize         = SR_MAXSIZE,
    binary_operators = ['+', '-', '*', '/'],
    unary_operators  = ['exp', 'sqrt', 'square'],
    parsimony        = 0.01,
    procs            = SR_PROCS,
    random_state     = RANDOM_STATE,
    verbosity        = 1,
    # output_jax_format=True,  # enable for JAX export
)

print(f'Running PySR on {len(X_sr):,} samples × {len(target_cols)} outputs ...')
sr_model.fit(X_sr, y_sr, variable_names=inlet_cols)
print('PySR done.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5b. BEST EQUATIONS TABLE
# ════════════════════════════════════════════════════════════════════════════
equations = []
for i, col in enumerate(target_cols):
    best = sr_model.get_best()[i]
    equations.append({
        'target':    col,
        'equation':  str(best['equation']),
        'latex':     sr_model.latex()[i] if hasattr(sr_model, 'latex') else '',
        'loss':      float(best['loss']),
        'complexity': int(best['complexity']),
    })

df_eqs = pd.DataFrame(equations)
print(df_eqs[['target', 'complexity', 'loss', 'equation']].to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6. EVALUATE: SR vs TEACHER NN
# ════════════════════════════════════════════════════════════════════════════
y_sr_pred = sr_model.predict(X_sr)  # (N, n_outputs)

metrics = []
for i, col in enumerate(target_cols):
    yt = y_distill[:, i]
    yp = y_sr_pred[:, i] if y_sr_pred.ndim == 2 else y_sr_pred
    r2  = r2_score(yt, yp)
    mae = mean_absolute_error(yt, yp)
    metrics.append({'target': col, 'R2_sr_vs_nn': r2, 'MAE_sr_vs_nn': mae})

df_metrics = pd.DataFrame(metrics)
print(df_metrics.to_string(index=False))
print(f'\nMean R²  (SR vs NN): {df_metrics.R2_sr_vs_nn.mean():.4f}')
print(f'Mean MAE (SR vs NN): {df_metrics.MAE_sr_vs_nn.mean():.4e}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 7. PARITY PLOTS: SR vs TEACHER NN
# ════════════════════════════════════════════════════════════════════════════
n_cols = 3
n_rows = int(np.ceil(len(target_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.8 * n_rows), squeeze=False)
setup_matplotlib(axes)

for i, col in enumerate(target_cols):
    ax  = axes[i // n_cols][i % n_cols]
    yt  = y_distill[:, i]
    yp  = y_sr_pred[:, i] if y_sr_pred.ndim == 2 else y_sr_pred
    r2  = df_metrics.loc[df_metrics.target == col, 'R2_sr_vs_nn'].values[0]

    ax.scatter(yt, yp, s=4, alpha=0.3, c='b', rasterized=True)
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, 'r-', lw=1)
    ax.set_title(f'{col}  R²={r2:.3f}', fontsize=9)
    ax.set_xlabel('NN teacher', fontsize=8)
    ax.set_ylabel('SR expression', fontsize=8)

for j in range(i + 1, n_rows * n_cols):
    axes[j // n_cols][j % n_cols].set_visible(False)

fig.suptitle('SR vs SimpleNN teacher (distillation parity)', fontsize=11)
plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'parity_sr_vs_nn.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8. VALIDATE SR ON CANTERA TEST DATA
# ════════════════════════════════════════════════════════════════════════════
# Compare SR expressions directly against Cantera exit-plane ground truth
# (inlet conditions only).

# Load raw exit-plane rows (last axial point per run, or rows where z/L == 1)
avail_cols = [c for c in target_cols if c in df_all.columns]
inlet_avail = [c for c in inlet_cols if c in df_all.columns]

if avail_cols and inlet_avail:
    df_exit = df_all[df_all.get('relative_position', pd.Series(0.0, index=df_all.index)) >= 0.99]
    if len(df_exit) == 0:
        df_exit = df_all.groupby(inlet_avail).last().reset_index()

    X_test_c = df_exit[inlet_avail].values
    y_test_c = df_exit[avail_cols].values
    y_sr_test = sr_model.predict(X_test_c)

    ct_metrics = []
    for i, col in enumerate(avail_cols):
        yt = y_test_c[:, i]
        yp = y_sr_test[:, i] if y_sr_test.ndim == 2 else y_sr_test
        ct_metrics.append({'target': col,
                           'R2_sr_vs_cantera': r2_score(yt, yp),
                           'MAE_sr_vs_cantera': mean_absolute_error(yt, yp)})
    df_ct = pd.DataFrame(ct_metrics)
    print(df_ct.to_string(index=False))
    print(f'\nMean R² (SR vs Cantera): {df_ct.R2_sr_vs_cantera.mean():.4f}')
else:
    print('[skip] Cantera validation: target columns not in training data file.')
    df_ct = pd.DataFrame()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8b. R² BAR CHART — SR fidelity
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
setup_matplotlib(axes)

# Left: SR vs NN
ax = axes[0]
ax.barh(df_metrics['target'], df_metrics['R2_sr_vs_nn'],
        facecolor='white', edgecolor='b', hatch='///')
ax.axvline(1.0, color='k', lw=0.8, ls='--')
ax.axvline(0.0, color='r', lw=0.8, ls='--', label='naive baseline')
ax.set_xlabel('R²')
ax.set_title('SR fidelity vs NN teacher')
ax.legend(fontsize=8)

# Right: SR vs Cantera (if available)
ax = axes[1]
if not df_ct.empty:
    ax.barh(df_ct['target'], df_ct['R2_sr_vs_cantera'],
            facecolor='white', edgecolor='r', hatch='///')
    ax.axvline(1.0, color='k', lw=0.8, ls='--')
    ax.axvline(0.0, color='r', lw=0.8, ls='--')
    ax.set_xlabel('R²')
    ax.set_title('SR accuracy vs Cantera ground truth')
else:
    ax.set_visible(False)

plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'r2_bar_sr_fidelity.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 9. DISPLAY EQUATIONS (LATEX)
# ════════════════════════════════════════════════════════════════════════════
from IPython.display import display, Math

for row in df_eqs.itertuples():
    print(f'--- {row.target} (complexity={row.complexity}, loss={row.loss:.4e}) ---')
    print(f'  Python: {row.equation}')
    if row.latex:
        display(Math(row.latex))
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10. EXPORT
# ════════════════════════════════════════════════════════════════════════════
if IF_SR_EXPORT:
    run_at = datetime.now().isoformat(timespec='seconds')

    export_meta = {
        'run_at':            run_at,
        'teacher_stem':      TEACHER_STEM,
        'teacher_class':     teacher.__class__.__name__,
        'is_profile_model':  IS_PROFILE_MODEL,
        'teacher_manifest':  str(manifest_path),
        'n_distill_samples': N_DISTILL_SAMPLES,
        'sr_params': {
            'niterations':    SR_N_ITERATIONS,
            'populations':    SR_POPULATIONS,
            'population_size': SR_POPULATION_SIZE,
            'maxsize':        SR_MAXSIZE,
        },
        'inlet_cols':  inlet_cols,
        'target_cols': target_cols,
        'equations':   equations,
        'metrics_sr_vs_nn': df_metrics.to_dict(orient='records'),
    }
    if not df_ct.empty:
        export_meta['metrics_sr_vs_cantera'] = df_ct.to_dict(orient='records')

    meta_path = EXPORT_DIR / f'{EXPORT_STEM}_manifest.json'
    with open(meta_path, 'w') as f:
        json.dump(export_meta, f, indent=2)
    print(f'[OK] manifest → {meta_path}')

    # Python module — one function per output
    py_path = EXPORT_DIR / f'{EXPORT_STEM}_equations.py'
    lines = [
        f'"""Auto-generated symbolic expressions (Main_8 PySR distillation, teacher={TEACHER_STEM})."""\n',
        'import numpy as np\n\n',
    ]
    for row in df_eqs.itertuples():
        fname = row.target.replace('.', '_').replace(' ', '_')
        args  = ', '.join(inlet_cols)
        lines.append(f'def predict_{fname}({args}):\n')
        lines.append(f'    return {row.equation}\n\n')
    with open(py_path, 'w') as f:
        f.writelines(lines)
    print(f'[OK] equations module → {py_path}')

    csv_path = EXPORT_DIR / f'{EXPORT_STEM}_metrics.csv'
    df_metrics.to_csv(csv_path, index=False)
    print(f'[OK] metrics → {csv_path}')

    print(f'\n=== Export summary (teacher: {TEACHER_STEM}) ===')
    print(f'  Mean R² (SR vs NN):      {df_metrics.R2_sr_vs_nn.mean():.4f}')
    if not df_ct.empty:
        print(f'  Mean R² (SR vs Cantera): {df_ct.R2_sr_vs_cantera.mean():.4f}')

---

## Summary

- **Teacher**: set by `TEACHER_STEM` — supports `SimpleNN` (Main_6) and `PINNPFR` (Main_7).
- **Student**: PySR symbolic expressions — one closed-form equation per output target.
- **Distillation**: teacher queried on N_DISTILL_SAMPLES inlet points sampled from the training data distribution. Profile models include z/L as an additional input.
- **Exports** (under `models/<sr_subdir>/`):
  - `*_manifest.json` — metadata + equations as strings
  - `*_equations.py` — callable Python functions (one per target, NumPy-compatible)
  - `*_metrics.csv` — R² and MAE vs teacher NN (and vs Cantera if available)

### Export stems by teacher
| `TEACHER_STEM` | Export directory | Equations file |
|---|---|---|
| `simple_nn_full_profile` | `models/sr_full_profile/` | `sr_full_profile_equations.py` |
| `pinn_pfr` | `models/sr_pinn/` | `sr_pinn_equations.py` |

### Operators
Binary: `+`, `-`, `*`, `/`. Unary: `exp`, `sqrt`, `square`.

### Tips for production
- Increase `SR_N_ITERATIONS` (≥ 500) and `N_DISTILL_SAMPLES` (≥ 20 000).
- Add `log` to `unary_operators` for species spanning decades.
- Use short `variable_names` aliases (`T_in`, `P_in`, …) for cleaner LaTeX.